# Cell 1 — Imports 

In [1]:
import os, sys, json, glob, hashlib, platform, datetime
from pathlib import Path

try:
    import pandas as pd
except Exception as e:
    print("pandas import failed, installing... error:", e)
    !pip -q install pandas
    import pandas as pd

# Cell 2 — Paths + output folder

In [2]:
KAGGLE_INPUT_DIR = Path("/kaggle/input")
KAGGLE_WORK_DIR  = Path("/kaggle/working")

OUT_DIR = KAGGLE_WORK_DIR / "output" / "phase6"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("KAGGLE_INPUT_DIR exists:", KAGGLE_INPUT_DIR.exists())
print("OUT_DIR:", OUT_DIR)

KAGGLE_INPUT_DIR exists: True
OUT_DIR: /kaggle/working/output/phase6


# Cell 3 — Helper functions (find files, hash, safe write)

In [3]:
def find_unique_file(patterns, search_root=KAGGLE_INPUT_DIR):
    """
    Find exactly one file matching any of the glob patterns under search_root.
    If multiple found, raise with list so you can choose explicitly.
    """
    matches = []
    for pat in patterns:
        matches.extend(glob.glob(str(search_root / "**" / pat), recursive=True))
    matches = sorted(set(matches))
    if len(matches) == 0:
        raise FileNotFoundError(f"No file found for patterns={patterns} under {search_root}")
    if len(matches) > 1:
        raise RuntimeError(f"Multiple files found for patterns={patterns}. "
                           f"Set path explicitly. Found:\n" + "\n".join(matches[:50]))
    return matches[0]

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    path = str(path)
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def now_utc_iso():
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

# Cell 4 — Load split CSVs + compute split summary

In [4]:
# Try auto-locating split CSVs
TRAIN_SPLIT_PATH = find_unique_file(["train_split.csv"], search_root=KAGGLE_INPUT_DIR)
VAL_SPLIT_PATH   = find_unique_file(["val_split.csv"],   search_root=KAGGLE_INPUT_DIR)

print("TRAIN_SPLIT_PATH:", TRAIN_SPLIT_PATH)
print("VAL_SPLIT_PATH:", VAL_SPLIT_PATH)

train_df = pd.read_csv(TRAIN_SPLIT_PATH)
val_df   = pd.read_csv(VAL_SPLIT_PATH)

def infer_has_crack_col(df):
    # Prefer explicit columns if they exist
    for c in ["has_crack", "has_crack_gt", "crack_present", "label", "is_crack"]:
        if c in df.columns:
            return c
    # Else: if an "empty_mask" style column exists, invert it
    for c in ["empty_mask", "is_empty_mask", "gt_empty"]:
        if c in df.columns:
            return c  # caller will invert
    return None

def summarize_split(df, name):
    n = len(df)
    cols = list(df.columns)
    col = infer_has_crack_col(df)

    crack_count = None
    no_crack_count = None
    crack_rule = None

    if col is None:
        crack_rule = "No has_crack/empty_mask column found; crack/no-crack counts not computed."
    else:
        if col in ["empty_mask", "is_empty_mask", "gt_empty"]:
            # assume 1 means empty => no crack
            crack_count = int((df[col] == 0).sum())
            no_crack_count = int((df[col] == 1).sum())
            crack_rule = f"has_crack = 1 - {col}"
        else:
            crack_count = int((df[col] == 1).sum())
            no_crack_count = int((df[col] == 0).sum())
            crack_rule = f"has_crack = {col}"

    # dataset/source hint
    source_hint = None
    if "image_path" in df.columns:
        sample_paths = df["image_path"].astype(str).head(50).tolist()
        source_hint = {
            "example_image_paths": sample_paths[:5],
            "mentions_sdnet": any("SDNET" in p.upper() or "DECK" in p.upper() or "PAVEMENT" in p.upper() or "WALL" in p.upper() for p in sample_paths),
            "mentions_masks": any("mask" in p.lower() for p in sample_paths),
        }

    return {
        "name": name,
        "n_samples": n,
        "columns": cols,
        "crack_count": crack_count,
        "no_crack_count": no_crack_count,
        "crack_rule": crack_rule,
        "source_hint": source_hint,
    }

split_summary = {
    "created_utc": now_utc_iso(),
    "train_split_path": TRAIN_SPLIT_PATH,
    "val_split_path": VAL_SPLIT_PATH,
    "train_split_sha256": sha256_file(TRAIN_SPLIT_PATH),
    "val_split_sha256": sha256_file(VAL_SPLIT_PATH),
    "train": summarize_split(train_df, "train"),
    "val": summarize_split(val_df, "val"),
}

write_json(OUT_DIR / "phase6_split_summary.json", split_summary)
print("Wrote:", OUT_DIR / "phase6_split_summary.json")
split_summary

TRAIN_SPLIT_PATH: /kaggle/input/datasets/arman00p/phase-3-csv/train_split.csv
VAL_SPLIT_PATH: /kaggle/input/datasets/arman00p/phase-3-csv/val_split.csv
Wrote: /kaggle/working/output/phase6/phase6_split_summary.json


/tmp/ipykernel_23/2082439112.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


{'created_utc': '2026-03-30T18:22:17Z',
 'train_split_path': '/kaggle/input/datasets/arman00p/phase-3-csv/train_split.csv',
 'val_split_path': '/kaggle/input/datasets/arman00p/phase-3-csv/val_split.csv',
 'train_split_sha256': '9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f1859d6682fb8e2065f',
 'val_split_sha256': 'e77305c80edf1a524274874537ef034d081a5047da0f23354cd94a85f73e8ccd',
 'train': {'name': 'train',
  'n_samples': 9603,
  'columns': ['stem',
   'image_path',
   'mask_path',
   'source',
   'empty_mask',
   'has_crack'],
  'crack_count': 8367,
  'no_crack_count': 1236,
  'crack_rule': 'has_crack = has_crack',
  'source_hint': {'example_image_paths': ['/kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset/images/forest_083.jpg',
    '/kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack_segmentation_dataset/images/CRACK500_20160328_151728_1281_361.jpg',
    '/kaggle/input/datasets/lakshaymiddha/crack-segmentation-dataset/crack

# Cell 5 — Repro config

In [5]:
repro_config = {
    "created_utc": now_utc_iso(),
    "seed": 7,
    "img_size_default": 448,
    "batch_size_default": 12,
    "lr_default": 2e-4,
    "weight_decay_default": 1e-4,
    "early_stopping_patience_default": 8,
    "threshold_baseline_default": 0.5,
    "min_component_area_detection_only": 30,
    "precision": "AMP (if CUDA)",
    "memory_format": "channels_last (if CUDA)",
    "matmul_precision": "high (if supported)",
    "notes": [
        "All comparisons must use the same val_split.csv (hash recorded).",
        "Report both overall metrics and crack-only metrics due to extreme pixel imbalance.",
        "Empty-aware IoU: if GT empty and Pred empty => IoU=1, else IoU=|A∩B|/|A∪B|.",
    ],
}

write_json(OUT_DIR / "phase6_repro_config.json", repro_config)
print("Wrote:", OUT_DIR / "phase6_repro_config.json")

Wrote: /kaggle/working/output/phase6/phase6_repro_config.json


/tmp/ipykernel_23/2082439112.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


# Cell 6 — Generate protocol.md

In [6]:
train_n = split_summary["train"]["n_samples"]
val_n   = split_summary["val"]["n_samples"]

train_cr = split_summary["train"]["crack_count"]
train_nc = split_summary["train"]["no_crack_count"]
val_cr   = split_summary["val"]["crack_count"]
val_nc   = split_summary["val"]["no_crack_count"]

protocol_md = f"""# Phase-6 Protocol — Multidataset Wall Crack Segmentation

Created (UTC): {now_utc_iso()}

## 1) Scope
We evaluate pixel-level crack segmentation models trained using PyTorch + segmentation-models-pytorch (SMP) with EfficientNet-B3 encoders.
All models must be evaluated on the **same exact split CSVs**, and split hashes are recorded for reproducibility.

## 2) Data splits (frozen)
- train_split.csv SHA256: {split_summary["train_split_sha256"]}
- val_split.csv   SHA256: {split_summary["val_split_sha256"]}

Counts:
- Train samples: {train_n}
  - crack-present: {train_cr}
  - no-crack: {train_nc}
  - rule used: {split_summary["train"]["crack_rule"]}
- Val samples: {val_n}
  - crack-present: {val_cr}
  - no-crack: {val_nc}
  - rule used: {split_summary["val"]["crack_rule"]}

## 3) Preprocessing (must match pretrained encoder)
If an encoder uses ImageNet pretrained weights, preprocessing must match the encoder's pretraining preprocessing.
SMP explicitly recommends preparing data “the same way as during weights pretraining” using `get_preprocessing_fn`.
Reference: SMP documentation (Quick Start).

## 4) Metrics (binary segmentation)
Let GT mask be A and predicted mask be B (binary sets of crack pixels).

### IoU (Jaccard / Intersection-over-Union)
IoU = |A ∩ B| / |A ∪ B|.

**Empty-aware IoU (required):**
- if A empty and B empty => IoU = 1
- if A empty and B non-empty => IoU = 0
- else IoU = |A ∩ B| / |A ∪ B|

### Dice (Sørensen–Dice)
Dice = 2|A ∩ B| / (|A| + |B|).
Use empty-aware handling consistent with IoU:
- if A empty and B empty => Dice = 1
- if A empty and B non-empty => Dice = 0
- else standard Dice.

### Crack-only vs Overall reporting (required)
Because cracks are thin and pixel-imbalanced:
- Report **overall** (all val images)
- Report **crack-only** (subset where GT has crack pixels)

### No-crack false positive rate (required)
On no-crack GT images (GT empty):
- FP if predicted mask has any positive pixels (pred_any = True)
- FP rate = FP_count / N_no_crack

### Pixel precision–recall AUC (required)
Compute pixel-level PR curve from sampled pixels using scikit-learn:
- `precision_recall_curve(y_true, y_score)`
- AUC computed over the PR curve points

Sampling:
- Use a fixed pixel sampling budget (e.g., 300,000 pixels) with deterministic seed.
- Store sampled size and seed in outputs.

## 5) Threshold policy
Baseline threshold for artifact parity: **0.5**.
Later (Phase-8), we will tune thresholds and publish `threshold_policy.json`.

## 6) Output naming convention (frozen)
All models must write the same artifact set with consistent prefixes, e.g.:
- *_val_metrics.csv
- *_val_summary_report.csv
- *_val_pr_auc_pixel.csv, *_pixel_pr_curve.png
- *_val_iou_hist_all.png, *_val_iou_hist_crack_only.png
- *_confidence_mean_pred_hist.png, *_confidence_p95_hist.png (p95 over predicted pixels only)
- *_gt_crack_area_hist.png, *_pred_crack_area_hist.png
- *_bbox_obb_overlays.png
- *_topk_worst_gallery.png, *_topk_worst_crack_only_gallery.png

## References (implementation guidance)
- SMP preprocessing recommendation (`get_preprocessing_fn`): https://segmentation-modelspytorch.readthedocs.io/ (Quick Start)
- scikit-learn PR curve: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_curve.html
- Jaccard/IoU definition: https://en.wikipedia.org/wiki/Jaccard_index
- Dice definition: https://en.wikipedia.org/wiki/Dice-S%C3%B8rensen_coefficient
"""

write_text(OUT_DIR / "phase6_protocol.md", protocol_md)
print("Wrote:", OUT_DIR / "phase6_protocol.md")

Wrote: /kaggle/working/output/phase6/phase6_protocol.md


/tmp/ipykernel_23/2082439112.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"


# Cell 7 — Runs manifest (registry CSV)

In [7]:
# Try to auto-find Model-1 checkpoint (from your screenshot name pattern)
# If multiple are found, set path manually.
def try_find_ckpt(patterns):
    try:
        return find_unique_file(patterns, search_root=KAGGLE_INPUT_DIR)
    except Exception as e:
        print("Checkpoint auto-find warning:", e)
        return None

model1_ckpt = try_find_ckpt(["/kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt"])
model2_ckpt = try_find_ckpt(["*deeplabv3plus*best*.pt"])
model3_ckpt = try_find_ckpt(["*unetplusplus*best*.pt", "*unetpp*best*.pt"])

def hash_if_exists(p):
    if p is None:
        return None
    if not os.path.exists(p):
        return None
    return sha256_file(p)

runs = [
    {
        "run_id": "baseline_model1_unet_effb3_eval256",
        "model": "Unet",
        "encoder": "EfficientNet-B3",
        "encoder_family": "efficientnet-b3 OR timm-efficientnet-b3 (auto-detect)",
        "eval_img_size": 256,
        "threshold": 0.5,
        "ckpt_path": model1_ckpt,
        "ckpt_sha256": hash_if_exists(model1_ckpt),
        "train_split_sha256": split_summary["train_split_sha256"],
        "val_split_sha256": split_summary["val_split_sha256"],
        "status": "planned (Phase-7)",
        "notes": "Phase-3R parity artifacts required for Model-1 at 256 and 448."
    },
    {
        "run_id": "baseline_model1_unet_effb3_eval448",
        "model": "Unet",
        "encoder": "EfficientNet-B3",
        "encoder_family": "efficientnet-b3 OR timm-efficientnet-b3 (auto-detect)",
        "eval_img_size": 448,
        "threshold": 0.5,
        "ckpt_path": model1_ckpt,
        "ckpt_sha256": hash_if_exists(model1_ckpt),
        "train_split_sha256": split_summary["train_split_sha256"],
        "val_split_sha256": split_summary["val_split_sha256"],
        "status": "planned (Phase-7)",
        "notes": "Resolution sensitivity must be documented."
    },
    {
        "run_id": "baseline_model2_deeplabv3plus_effb3_eval448",
        "model": "DeepLabV3Plus",
        "encoder": "EfficientNet-B3",
        "encoder_family": "timm-efficientnet-b3 (as used in your runs)",
        "eval_img_size": 448,
        "threshold": 0.5,
        "ckpt_path": model2_ckpt,
        "ckpt_sha256": hash_if_exists(model2_ckpt),
        "train_split_sha256": split_summary["train_split_sha256"],
        "val_split_sha256": split_summary["val_split_sha256"],
        "status": "completed (Phase-5)",
        "notes": "Has confidence/crack-area hist + detection exports already."
    },
    {
        "run_id": "baseline_model3_unetplusplus_effb3_eval448",
        "model": "UnetPlusPlus",
        "encoder": "EfficientNet-B3",
        "encoder_family": "timm-efficientnet-b3 (as used in your runs)",
        "eval_img_size": 448,
        "threshold": 0.5,
        "ckpt_path": model3_ckpt,
        "ckpt_sha256": hash_if_exists(model3_ckpt),
        "train_split_sha256": split_summary["train_split_sha256"],
        "val_split_sha256": split_summary["val_split_sha256"],
        "status": "completed (Phase-5)",
        "notes": "Has confidence/crack-area hist + detection exports already."
    },
]

runs_df = pd.DataFrame(runs)
runs_df.to_csv(OUT_DIR / "phase6_runs_manifest.csv", index=False)
print("Wrote:", OUT_DIR / "phase6_runs_manifest.csv")
runs_df

Wrote: /kaggle/working/output/phase6/phase6_runs_manifest.csv


,run_id,model,encoder,encoder_family,eval_img_size,threshold,ckpt_path,ckpt_sha256,train_split_sha256,val_split_sha256,status,notes
0,baseline_model1_unet_effb3_eval256,Unet,EfficientNet-B3,efficientnet-b3 OR timm-efficientnet-b3 (auto-...,256,0.5,/kaggle/input/datasets/arman00p/model-1-checkp...,48c0df9eef4b610fcbacd535f881a671b82c57f25ceabf...,9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f...,e77305c80edf1a524274874537ef034d081a5047da0f23...,planned (Phase-7),Phase-3R parity artifacts required for Model-1...
1,baseline_model1_unet_effb3_eval448,Unet,EfficientNet-B3,efficientnet-b3 OR timm-efficientnet-b3 (auto-...,448,0.5,/kaggle/input/datasets/arman00p/model-1-checkp...,48c0df9eef4b610fcbacd535f881a671b82c57f25ceabf...,9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f...,e77305c80edf1a524274874537ef034d081a5047da0f23...,planned (Phase-7),Resolution sensitivity must be documented.
2,baseline_model2_deeplabv3plus_effb3_eval448,DeepLabV3Plus,EfficientNet-B3,timm-efficientnet-b3 (as used in your runs),448,0.5,/kaggle/input/datasets/arman00p/model-2-checkp...,2b471d69581e7ebbda88e5e942dd6eddd6fb3a313976e0...,9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f...,e77305c80edf1a524274874537ef034d081a5047da0f23...,completed (Phase-5),Has confidence/crack-area hist + detection exp...
3,baseline_model3_unetplusplus_effb3_eval448,UnetPlusPlus,EfficientNet-B3,timm-efficientnet-b3 (as used in your runs),448,0.5,/kaggle/input/datasets/arman00p/model-3-checkp...,ee06130dddcb039626c1dfcf9ab6ec300676dc56a32933...,9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f...,e77305c80edf1a524274874537ef034d081a5047da0f23...,completed (Phase-5),Has confidence/crack-area hist + detection exp...


# Cell 8 — Quick sanity print + what Phase-7 will consume

In [8]:
print("Phase-6 complete.")
print("Created files:")
for p in sorted(OUT_DIR.glob("*")):
    print(" -", p.name)

print("\nPhase-7 will require these inputs (already attached in your screenshot):")
print(" - Crack Segmentation Dataset (images + masks)")
print(" - phase-3 CSV: train_split.csv / val_split.csv")
print(" - Model-1 checkpoint: unet_efficientnetb3_best_*.pt")
print(" - SDNET2018 only if split CSV paths reference SDNET images")

Phase-6 complete.
Created files:
 - phase6_protocol.md
 - phase6_repro_config.json
 - phase6_runs_manifest.csv
 - phase6_split_summary.json

Phase-7 will require these inputs (already attached in your screenshot):
 - Crack Segmentation Dataset (images + masks)
 - phase-3 CSV: train_split.csv / val_split.csv
 - Model-1 checkpoint: unet_efficientnetb3_best_*.pt
 - SDNET2018 only if split CSV paths reference SDNET images


# fix the manifest Model-1 ckpt info

In [9]:
import pandas as pd
from pathlib import Path
import hashlib

OUT_DIR = Path("/kaggle/working/output/phase6")
manifest_path = OUT_DIR / "phase6_runs_manifest.csv"


MODEL1_CKPT = "/kaggle/input/datasets/arman00p/model-1-checkpoints-unet-efficientnet-b3/output/SAVED_MODELS/unet_efficientnetb3_best_ep029_20260219_205815.pt"

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

df = pd.read_csv(manifest_path)

mask = df["run_id"].astype(str).str.contains("baseline_model1_unet_effb3", na=False)
df.loc[mask, "ckpt_path"] = MODEL1_CKPT
df.loc[mask, "ckpt_sha256"] = sha256_file(MODEL1_CKPT)

df.to_csv(manifest_path, index=False)
print("Updated:", manifest_path)
df[mask]

Updated: /kaggle/working/output/phase6/phase6_runs_manifest.csv


,run_id,model,encoder,encoder_family,eval_img_size,threshold,ckpt_path,ckpt_sha256,train_split_sha256,val_split_sha256,status,notes
0,baseline_model1_unet_effb3_eval256,Unet,EfficientNet-B3,efficientnet-b3 OR timm-efficientnet-b3 (auto-...,256,0.5,/kaggle/input/datasets/arman00p/model-1-checkp...,48c0df9eef4b610fcbacd535f881a671b82c57f25ceabf...,9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f...,e77305c80edf1a524274874537ef034d081a5047da0f23...,planned (Phase-7),Phase-3R parity artifacts required for Model-1...
1,baseline_model1_unet_effb3_eval448,Unet,EfficientNet-B3,efficientnet-b3 OR timm-efficientnet-b3 (auto-...,448,0.5,/kaggle/input/datasets/arman00p/model-1-checkp...,48c0df9eef4b610fcbacd535f881a671b82c57f25ceabf...,9c02ef25f7cd668418a76fd556d0a8886258eaa7f6418f...,e77305c80edf1a524274874537ef034d081a5047da0f23...,planned (Phase-7),Resolution sensitivity must be documented.
